In [ ]:
# prompt: x,y,z from this file /content/subject-1.csv  in np

import numpy as np
import pandas as pd

df = pd.read_csv('/content/comparison.csv')
x = df['true_x'].to_numpy()
y = df['true_y'].to_numpy()
z = df['true_z'].to_numpy()



In [ ]:
import numpy as np
import plotly.graph_objects as go

# Number of points in the trajectory
n_points = len(z)

# Generate a smooth trajectory (e.g., sine and cosine curves)
#x = np.linspace(0, 10, n_points)
#y = np.sin(x) + np.random.normal(0, 0.1, n_points)  # Sine wave with some noise
#z = np.cos(x) + np.random.normal(0, 0.1, n_points)  # Cosine wave with some noise


# Generate flow score (F) based on trajectory
# For simplicity, let's assume that F is higher when the trajectory is "higher" (more positive)
F = np.interp(y, (y.min(), y.max()), (0, 1))  # Normalize y values to the range [0, 1]

# Generate optimality score (O) based on the smoothness of the trajectory
# A less smooth trajectory (more noise) will have lower O, and a smooth trajectory will have higher O
O = 1 - np.interp(np.abs(np.diff(y)), (0, np.max(np.abs(np.diff(y)))), (0, 1))
O = np.concatenate(([1], O))  # Add 1 at the start to match the length

# Create an RGB color array based on F and O
colors = np.zeros((n_points, 3))  # Array to hold RGB values
colors[:, 0] = 1 - F  # Red component (1 for low F, 0 for high F)
colors[:, 1] = F      # Green component (0 for low F, 1 for high F)
colors[:, 2] = O      # Blue component (1 for high O, 0 for low O)

# Create the 3D plot
trace = go.Scatter3d(
    x=x,
    y=y,
    z=z,
    mode='markers+lines',
    marker=dict(
        size=5,
        color=colors,  # RGB values
        opacity=0.8
    ),
    line=dict(color='black', width=1)
)

layout = go.Layout(
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z'
    ),
    title="Real-Time Trajectory Visualization with Noise"
)

fig = go.Figure(data=[trace], layout=layout)
fig.show()

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Number of points in the trajectory
n_points = len(z)

# Generate a smooth trajectory (e.g., sine and cosine curves)
#x = np.linspace(0, 10, n_points) + np.random.normal(0, 0.01, n_points)
#y = np.sin(x) + np.random.normal(0, 0.01, n_points)  # Sine wave with some noise
#z = np.cos(x) + np.random.normal(0, 0.01, n_points)  # Cosine wave with some noise

# Generate flow score (F) based on trajectory
#F = np.interp(y, (y.min(), y.max()), (0, 1))  # Normalize y values to the range [0, 1]

# Generate optimality score (O) based on smoothness
#O = 1 - np.interp(np.abs(np.diff(y)), (0, np.max(np.abs(np.diff(y)))), (0, 1))
#O = np.concatenate(([1], O))  # Add 1 at the start to match the length

# Compute rewards based on proximity to ideal movement (F=1, O=max)
rewards = 200 * F + O - 100  # Centering rewards around 0
rewards = np.round(rewards, 2)  # Round for readability

# Create an RGB color array based on F and O
colors = np.zeros((n_points, 3))  # Array to hold RGB values
colors[:, 0] = 1 - F  # Red component (1 for low F, 0 for high F)
colors[:, 1] = F      # Green component (0 for low F, 1 for high F)
colors[:, 2] = O      # Blue component (1 for high O, 0 for low O)

# Create the 3D scatter plot
trace = go.Scatter3d(
    x=x,
    y=y,
    z=z,
    mode='markers+lines',
    marker=dict(
        size=5,
        color=colors,  # RGB values
        opacity=0.8
    ),
    line=dict(color='black', width=1)
)

# Add annotations for key points with reward values
annotations = [
    dict(
        x=x[i], y=y[i], z=z[i],
        text=f"Reward: {rewards[i]}",
        showarrow=True,
        arrowhead=1,
        ax=40, ay=-40
    )
    for i in range(0, n_points, 40)  # Annotate every 10 points
]

layout = go.Layout(
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',  # Correct way to add Z-axis title
        annotations=annotations  # Add annotations
    ),
    title="Wighted Real-Time Trajectory Visualization",
    width=1000,  # Increase width
    height=800   # Increase height
)


fig = go.Figure(data=[trace], layout=layout)
fig.show()


In [ ]:
# prompt: Save the data in a dataframe with the x point, flowscore, optimality and reward

import pandas as pd

# Assuming x, F, O, and rewards are already defined from the previous code



df = pd.DataFrame({'x': x, 'y':y, 'z':z, 'flowscore': F, 'optimality': O, 'reward': rewards})
df


In [ ]:

!pip install stable-baselines3


In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import plotly.graph_objects as go

class ConvDiffusionPolicy(nn.Module):
    def __init__(self, state_channels, state_dim, action_dim, hidden_dim=256):
        super().__init__()
        self.state_channels = state_channels
        self.state_dim = state_dim  # This is now per channel
        self.action_dim = action_dim

        # Convolutional layers for feature extraction
        self.conv_layers = nn.Sequential(
            nn.Conv1d(state_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),  # Reduce dimension by half
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
        )

        # Calculate the flattened size after convolutions
        self.flat_size = 64 * (state_dim // 2)

        # Fully connected layers for action prediction
        self.fc_layers = nn.Sequential(
            nn.Linear(self.flat_size, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, action_dim)
        )

    def forward(self, states):
        """
        Forward pass to predict optimal actions from multi-dimensional states

        Args:
            states: Tensor of shape [batch_size, state_channels, state_dim]
        """
        # Apply convolutions to extract features
        x = self.conv_layers(states)

        # Flatten the convolutional features
        x = x.view(x.size(0), -1)

        # Apply fully connected layers for action prediction
        return self.fc_layers(x)

    def reshape_input(self, states):
        """Helper to reshape flat state input into proper CNN format"""
        if len(states.shape) == 2:  # Batch of flat states
            return states.view(states.size(0), self.state_channels, -1)
        elif len(states.shape) == 1:  # Single flat state
            return states.view(1, self.state_channels, -1)

class ConvMultiDimensionalDataset(Dataset):
    def __init__(self, df, state_features, action_features, state_channels=3, state_dim_per_channel=10):
        """
        Initialize the dataset with multi-dimensional state representation

        Args:
            df: DataFrame containing demonstration data
            state_features: List of column names for state features
            action_features: List of column names for action features
            state_channels: Number of channels for state representation
            state_dim_per_channel: Dimension size per channel
        """
        self.state_channels = state_channels
        self.state_dim_per_channel = state_dim_per_channel
        self.state_features = state_features
        self.action_features = action_features

        # Extract original state and action values
        self.original_states = df[state_features].values
        self.original_actions = df[action_features].values

        # Calculate statistics for normalization
        self.state_mean = np.mean(self.original_states, axis=0)
        self.state_std = np.std(self.original_states, axis=0) + 1e-6  # Prevent division by zero
        self.action_mean = np.mean(self.original_actions, axis=0)
        self.action_std = np.std(self.original_actions, axis=0) + 1e-6

        # Normalize the data
        self.normalized_states = (self.original_states - self.state_mean) / self.state_std
        self.normalized_actions = (self.original_actions - self.action_mean) / self.action_std

        # Convert to tensors
        self.states = torch.FloatTensor(self.normalized_states)
        self.actions = torch.FloatTensor(self.normalized_actions)

        # Reshape states to match CNN input format [batch, channels, features_per_channel]
        self._reshape_states()

    def _reshape_states(self):
        """Reshape flat states into CNN format"""
        batch_size = len(self.states)
        total_features = len(self.state_features)

        # If we have too few features, pad with zeros
        if total_features < self.state_channels * self.state_dim_per_channel:
            padding_size = self.state_channels * self.state_dim_per_channel - total_features
            padding = torch.zeros(batch_size, padding_size)
            self.states = torch.cat([self.states, padding], dim=1)

        # Reshape to [batch, channels, features_per_channel]
        self.states = self.states.view(batch_size, self.state_channels, -1)

        # Ensure each channel has the right dimension
        if self.states.shape[2] != self.state_dim_per_channel:
            # Interpolate to the correct dimension
            self.states = F.interpolate(
                self.states,
                size=self.state_dim_per_channel,
                mode='linear',
                align_corners=False
            )

    def __len__(self):
        return len(self.original_states)

    def __getitem__(self, idx):
        return self.states[idx], self.actions[idx]

    def normalize_state(self, state):
        """Normalize a single state and reshape for CNN"""
        if isinstance(state, np.ndarray):
            normalized = (state - self.state_mean) / self.state_std
            state_tensor = torch.FloatTensor(normalized)
        else:
            normalized = (state - self.state_mean) / self.state_std
            state_tensor = normalized

        # Reshape for CNN - add batch and channel dimensions if needed
        if len(state_tensor.shape) == 1:  # Single flat state
            # Pad if necessary
            if len(state_tensor) < self.state_channels * self.state_dim_per_channel:
                padding_size = self.state_channels * self.state_dim_per_channel - len(state_tensor)
                padding = torch.zeros(padding_size)
                state_tensor = torch.cat([state_tensor, padding])

            # Reshape to [1, channels, features_per_channel]
            state_tensor = state_tensor.view(1, self.state_channels, -1)

            # Ensure each channel has the right dimension
            if state_tensor.shape[2] != self.state_dim_per_channel:
                state_tensor = F.interpolate(
                    state_tensor,
                    size=self.state_dim_per_channel,
                    mode='linear',
                    align_corners=False
                )

        return state_tensor

    def denormalize_action(self, action):
        """Denormalize a predicted action"""
        if isinstance(action, np.ndarray):
            action = torch.FloatTensor(action)
        return action * self.action_std + self.action_mean

class ConvDiffusionEnv(gym.Env):
    def __init__(self, df, state_features, action_features, dataset=None):
        super().__init__()
        self.df = df
        self.state_features = state_features
        self.action_features = action_features
        self.current_step = 0
        self.dataset = dataset  # Store dataset for normalization
        self.max_steps = len(df) - 1

        # For tracking performance
        self.total_reward = 0
        self.rewards_history = []

        # Define action and observation spaces
        # Action space - for each action feature
        self.action_space = spaces.Box(
            low=np.array([float(df[feat].min()) for feat in action_features]),
            high=np.array([float(df[feat].max()) for feat in action_features]),
            shape=(len(action_features),),
            dtype=np.float32
        )

        # Observation space - for each state feature
        self.observation_space = spaces.Box(
            low=np.array([float(df[feat].min()) for feat in state_features]),
            high=np.array([float(df[feat].max()) for feat in state_features]),
            shape=(len(state_features),),
            dtype=np.float32
        )

    def reset(self, seed=None):
        if seed is not None:
            super().reset(seed=seed)
        self.current_step = 0
        self.total_reward = 0
        self.rewards_history = []

        # Get initial state
        state = np.array([
            self.df[feat].iloc[0] for feat in self.state_features
        ], dtype=np.float32)

        return state, {}

    def step(self, action):
        # Get the target reward/action from the dataset
        target_actions = np.array([self.df[feat].iloc[self.current_step] for feat in self.action_features])

        # Calculate reward - combination of direct reward and closeness to demonstration
        reward = np.mean(action)  # Basic reward is the mean of action values

        # Add a bonus if we're close to the optimal action from the dataset
        error = np.mean(np.abs(action - target_actions))
        max_error = np.mean(np.abs(self.action_space.high - self.action_space.low))
        closeness_factor = 1.0 - min(1.0, error / (max_error + 1e-8))
        reward_bonus = closeness_factor * 0.5  # Scale the bonus

        # The final reward combines direct reward maximization with alignment to demonstrations
        final_reward = reward + reward_bonus

        self.total_reward += final_reward
        self.rewards_history.append(final_reward)

        # Increment step
        self.current_step += 1
        done = self.current_step >= self.max_steps

        if done:
            self.current_step = self.max_steps  # Cap at last index

        # Get next state
        next_state = np.array([
            self.df[feat].iloc[self.current_step] for feat in self.state_features
        ], dtype=np.float32)

        return next_state, final_reward, done, False, {}

def train_conv_policy(df, state_features, action_features,
                     state_channels=3, state_dim_per_channel=10,
                     n_epochs=100, batch_size=64, learning_rate=1e-3):

    dataset = ConvMultiDimensionalDataset(
        df,
        state_features,
        action_features,
        state_channels=state_channels,
        state_dim_per_channel=state_dim_per_channel
    )
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # Create a convolutional policy that maps states to optimal actions
    policy = ConvDiffusionPolicy(
        state_channels=state_channels,
        state_dim=state_dim_per_channel,
        action_dim=len(action_features),
        hidden_dim=256
    )

    optimizer = torch.optim.Adam(policy.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, verbose=True
    )

    loss_history = []
    best_loss = float('inf')
    best_policy = None

    for epoch in tqdm(range(n_epochs), desc="Training"):
        total_loss = 0

        for states, actions in dataloader:
            optimizer.zero_grad()

            # Forward pass - predict actions directly from states
            pred_actions = policy(states)

            # Main loss: MSE to learn from demonstrations
            loss = F.mse_loss(pred_actions, actions)

            # Backward pass
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        loss_history.append(avg_loss)

        # Update learning rate based on loss
        scheduler.step(avg_loss)

        # Save best model
        if avg_loss < best_loss:
            best_loss = avg_loss
            best_policy = policy.state_dict().copy()

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{n_epochs}, Loss: {avg_loss:.6f}, LR: {optimizer.param_groups[0]['lr']:.6f}")

    # Load best model
    if best_policy is not None:
        policy.load_state_dict(best_policy)

    return policy, loss_history, dataset

def evaluate_conv_policy(policy, env, dataset, num_episodes=5):
    total_rewards = []
    all_trajectories = []

    for episode in range(num_episodes):
        state, _ = env.reset()
        done = False
        episode_states = [state]
        episode_actions = []
        episode_rewards = []

        print(f"\nStarting Episode {episode + 1}")

        step = 0
        while not done:
            # Normalize state using dataset statistics
            state_tensor = dataset.normalize_state(state)

            # Get action from policy
            with torch.no_grad():
                action_tensor = policy(state_tensor)

            # Denormalize action
            action = dataset.denormalize_action(action_tensor).numpy().flatten()

            # Clip action to environment bounds
            action = np.clip(
                action,
                env.action_space.low,
                env.action_space.high
            )

            # Step environment
            next_state, reward, done, _, info = env.step(action)

            print(f"Step {step}: State shape={state.shape}, Action={action.round(4)}, Reward={reward:.4f}")

            # Save for trajectory
            episode_states.append(next_state)
            episode_actions.append(action)
            episode_rewards.append(reward)

            # Update state
            state = next_state
            step += 1

        # Calculate total episode reward
        episode_total_reward = sum(episode_rewards)
        total_rewards.append(episode_total_reward)

        # Save trajectory
        all_trajectories.append({
            'states': np.array(episode_states),
            'actions': np.array(episode_actions),
            'rewards': np.array(episode_rewards)
        })

        print(f"Episode {episode + 1} Total Reward: {episode_total_reward:.4f}")

    # Plot rewards
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, num_episodes + 1), total_rewards, marker='o', linestyle='-')
    plt.title('Episode Rewards')
    plt.xlabel('Episode')
    plt.ylabel('Total Reward')
    plt.grid(True)
    plt.show()

    # Plot rewards over time for each episode
    plt.figure(figsize=(12, 6))
    for idx, traj in enumerate(all_trajectories):
        plt.plot(traj['rewards'], marker='.', label=f'Episode {idx + 1}')

    plt.title('Rewards Over Time')
    plt.xlabel('Step')
    plt.ylabel('Reward')
    plt.legend()
    plt.grid(True)
    plt.show()

    return np.mean(total_rewards), np.std(total_rewards), all_trajectories

def create_and_train_conv_model(df, state_features, action_features, state_channels=3, state_dim_per_channel=10):
    # Train the convolutional policy
    policy, loss_history, dataset = train_conv_policy(
        df,
        state_features,
        action_features,
        state_channels=state_channels,
        state_dim_per_channel=state_dim_per_channel,
        n_epochs=10,
        batch_size=64,
        learning_rate=1e-3
    )

    # Create environment
    env = ConvDiffusionEnv(df, state_features, action_features, dataset)

    # Plot training loss
    plt.figure(figsize=(10, 6))
    plt.plot(loss_history)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training Loss')
    plt.grid(True)
    plt.show()

    return policy, env, dataset, loss_history



In [ ]:
# Example usage:
state_features = ['x', 'y', 'z', 'flowscore', 'optimality']
action_features = ['reward']
policy, env, dataset, loss_history = create_and_train_conv_model(df, state_features, action_features)

# Rename keys
new_state_dict = {}
for key, value in policy.state_dict().items():
    new_key = key.replace("conv_layers.", "conv")  # Example renaming
    new_key = new_key.replace("fc_layers.", "fc")
    new_state_dict[new_key] = value



# Save the trained policy model
torch.save(new_state_dict, "trained_conv_policy.pth")
print("Model saved as trained_conv_policy.pth")

In [ ]:
# Evaluate the policy
mean_reward, std_reward, trajectories = evaluate_conv_policy(policy, env, dataset, num_episodes=10)

# Convert trajectories to a DataFrame
trajectory_data = []
for episode_idx, trajectory in enumerate(trajectories):
    states = trajectory["states"]
    actions = trajectory["actions"]
    rewards = trajectory["rewards"]

    # Debugging prints
    print(f"Episode {episode_idx}:")
    print(f"  States: {len(states)}")
    print(f"  Rewards: {len(rewards)}")
    print(f"  Actions: {len(actions)}")

    # Ensure the loop does not go out of bounds
    min_steps = min(len(states) - 1, len(rewards), len(actions))

    for step_idx in range(min_steps):
        trajectory_data.append({
            "x": states[step_idx][0],
            "y": states[step_idx][1],
            "z": states[step_idx][2],
            "reward": rewards[step_idx],
            "action": actions[step_idx][0]  # Assuming actions are lists
        })

# Create DataFrame and save as CSV
df_trajectories = pd.DataFrame(trajectory_data)
df_trajectories.to_csv("evaluated_trajectories.csv", index=False)

print(df_trajectories.head())
print("Trajectories saved as evaluated_trajectories.csv")

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np

# Load the data from the CSV file
df_trajectories = pd.read_csv("evaluated_trajectories.csv")

print(df_trajectories.head())

# Kalman Filter Implementation
def kalman_filter(data, process_var=1e-5, measurement_var=1e-2):
    """
    Applies a simple 1D Kalman filter to smooth the data.
    :param data: List or NumPy array of noisy measurements.
    :param process_var: Estimated process variance (small positive value).
    :param measurement_var: Estimated measurement variance (small positive value).
    :return: Smoothed data array.
    """
    n = len(data)
    x_hat = np.zeros(n)  # Estimated value
    P = np.zeros(n)      # Error covariance

    # Initial estimates
    x_hat[0] = data[0]
    P[0] = 1.0

    for k in range(1, n):
        # Prediction step
        x_hat[k] = x_hat[k - 1]
        P[k] = P[k - 1] + process_var

        # Update step
        K = P[k] / (P[k] + measurement_var)  # Kalman gain
        x_hat[k] = x_hat[k] + K * (data[k] - x_hat[k])  # Update estimate
        P[k] = (1 - K) * P[k]  # Update error covariance

    return x_hat

# Apply Kalman filter to trajectory data
df_trajectories['x'] = kalman_filter(df_trajectories['x'].values)
df_trajectories['y'] = kalman_filter(df_trajectories['y'].values)
df_trajectories['z'] = kalman_filter(df_trajectories['z'].values)

# Normalize reward and action values for RGB scaling
reward_min, reward_max = df_trajectories['reward'].min(), df_trajectories['reward'].max()
action_abs_max = df_trajectories['action'].abs().max()

# Function to compute RGB color
def compute_rgb(reward, action):
    # Normalize reward to range [0, 255] for red/green components
    norm_reward = (reward - reward_min) / (reward_max - reward_min)  # Normalize to [0,1]
    g = int(norm_reward * 255) if reward > 0 else 0  # Green intensity for positive rewards
    r = int((1 - norm_reward) * 255) if reward < 0 else 0  # Red intensity for negative rewards

    # Normalize action magnitude to [0, 255] for blue component
    norm_action = abs(action) / action_abs_max
    b = int(norm_action * 255)

    return f'rgb({r},{g},{b})'  # Convert to RGB format

# Apply the color function
df_trajectories['color'] = df_trajectories.apply(lambda row: compute_rgb(row['reward'], row['action']), axis=1)

# Determine axis limits for proper scaling
x_min, x_max = df_trajectories['x'].min(), df_trajectories['x'].max()
y_min, y_max = df_trajectories['y'].min(), df_trajectories['y'].max()
z_min, z_max = df_trajectories['z'].min(), df_trajectories['z'].max()

# Create a 3D scatter plot with computed colors
fig = go.Figure(data=go.Scatter3d(
    x=df_trajectories['x'],
    y=df_trajectories['y'],
    z=df_trajectories['z'],
    marker=dict(
        size=5,
        color=df_trajectories['color'],  # Set color dynamically
        opacity=1.0
    )
))

# Update layout with correct scaling
fig.update_layout(
    scene=dict(
        xaxis=dict(title='X', range=[x_min, x_max]),
        yaxis=dict(title='Y', range=[y_min, y_max]),
        zaxis=dict(title='Z', range=[z_min, z_max])
    ),
    title='3D Trajectory with Kalman Filter (Color by Reward & Action)'
)

fig.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Confusion matrix data
cm = np.array([
    [9292,  3917,  5141],
    [ 802, 27153,  6079],
    [1916,  8009, 25544]
])

# Class names
classes = ['rest', 'oddball', 'msit']

fig, ax = plt.subplots()
# Display the matrix with a blue-white colormap
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')

# Add a colorbar
cbar = ax.figure.colorbar(im, ax=ax)
cbar.ax.set_ylabel('Count', rotation=-90, va="bottom")

# Set axis labels and ticks
ax.set(
    xticks=np.arange(len(classes)),
    yticks=np.arange(len(classes)),
    xticklabels=classes,
    yticklabels=classes,
    xlabel='Predicted Label',
    ylabel='True Label',
    title='Confusion Matrix'
)

# Rotate the x tick labels for readability
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

# Annotate each cell with its value
threshold = cm.max() / 2
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        color = "white" if cm[i, j] > threshold else "black"
        ax.text(
            j, i, f"{cm[i, j]:,}",
            ha="center", va="center",
            color=color
        )

fig.tight_layout()
plt.show()
